In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [2]:
# --- Load data ---
df = pd.read_excel("Concrete_Data.xls")
X = df.drop(columns=['Concrete compressive strength(MPa, megapascals) '])
y = df['Concrete compressive strength(MPa, megapascals) '].values

# --- Split into training and test set ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Standardize whole training set (for final evaluation later) ---
scaler_full = StandardScaler()
X_train_std = scaler_full.fit_transform(X_train)
X_test_std = scaler_full.transform(X_test)

In [3]:
lambdas = np.logspace(-4, 4, 100)      # Ridge hyperparameter
h_values = [1, 2, 3, 4]           # Number of hidden units

# --- Nested CV settings ---
K1 = K2 = 10
outer_cv = KFold(n_splits=K1, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=K2, shuffle=True, random_state=42)

# Define once, OUTSIDE the outer loop — they refit per inner split
ridge_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(random_state=42))
])

mlp_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(16,),     
        max_iter=2000,
        early_stopping=True,
        random_state=42
    ))
])


ridge_gs = GridSearchCV(
    estimator=ridge_pipe,
    param_grid={"ridge__alpha": lambdas},
    scoring="neg_mean_squared_error",
    cv=inner_cv,
    n_jobs=-1
)

ann_gs = GridSearchCV(
    estimator=mlp_pipe,
    param_grid={"mlp__hidden_layer_sizes": [(h,) for h in h_values]},
    scoring="neg_mean_squared_error",
    cv=inner_cv,
    n_jobs=-1
)

# --- Store results ---
results = []
ann_loss_curves = []

# --- Outer CV ---
fold_number = 1
for train_idx, test_idx in outer_cv.split(X_train, y_train):
    # Split original (unscaled) training data
    X_train_outer, X_test_outer = X_train.iloc[train_idx], X_train.iloc[test_idx]
    y_train_outer, y_test_outer = y_train[train_idx], y_train[test_idx]

    # Baseline
    baseline_pred = np.full_like(y_test_outer, y_train_outer.mean(), dtype=float)
    baseline_mse = mean_squared_error(y_test_outer, baseline_pred)

    # Ridge
    ridge_gs.fit(X_train_outer, y_train_outer)
    best_lambda = ridge_gs.best_params_["ridge__alpha"]
    ridge_test_mse = mean_squared_error(y_test_outer, ridge_gs.best_estimator_.predict(X_test_outer))

    # ANN
    ann_gs.fit(X_train_outer, y_train_outer)
    best_h = ann_gs.best_params_["mlp__hidden_layer_sizes"][0]
    ann_test_mse = mean_squared_error(y_test_outer, ann_gs.best_estimator_.predict(X_test_outer))

    # Store results
    results.append({
        'Fold': fold_number,
        'λ*': best_lambda,
        'h*': best_h,
        'Ridge Test MSE': ridge_test_mse,
        'ANN Test MSE': ann_test_mse,
        'Baseline MSE': baseline_mse
    })

    print(f"Fold {fold_number}: λ*={best_lambda:.4f}, h*={best_h}, "
          f"Ridge MSE={ridge_test_mse:.3f}, ANN MSE={ann_test_mse:.3f}, "
          f"Baseline MSE={baseline_mse:.3f}")

    fold_number += 1

# --- Display results table ---
results_df = pd.DataFrame(results).sort_values("Fold")
display(results_df)

print("\nAverage Test MSEs (Outer CV):")
print(results_df[['Ridge Test MSE', 'ANN Test MSE', 'Baseline MSE']].mean())

# --- Plot ANN loss curves ---
plt.figure(figsize=(10,6))
for i, loss_curve in enumerate(ann_loss_curves):
    plt.plot(loss_curve, label=f'Fold {i+1}')
plt.xlabel('Iteration')
plt.ylabel('Training Loss (MSE)')
plt.title('ANN Training Loss per Outer Fold')
plt.legend()
plt.grid(True)
plt.show()

# --- Final evaluation on untouched test set ---
best_lambda_global = results_df['λ*'].mode()[0]
best_h_global = results_df['h*'].mode()[0]
# best_neurons_global = [16*(2**i) for i in range(best_h_global)]

# Baseline
baseline_rmse = np.sqrt(mean_squared_error(y_test, np.full_like(y_test, y_train.mean(), dtype=float)))

# Ridge
ridge_final_global = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=best_lambda_global, random_state=42))]).fit(X_train, y_train)
ridge_test_rmse = np.sqrt(mean_squared_error(y_test, ridge_final_global.predict(X_test)))

# ANN
ann_final_global = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPRegressor(
        hidden_layer_sizes=(best_h_global,),
        max_iter=3000,
        early_stopping=True,
        random_state=42
    ))
]).fit(X_train, y_train)
ann_test_rmse = np.sqrt(mean_squared_error(y_test, ann_final_global.predict(X_test)))

print(f"\nFinal Evaluation on Held-out Test Set:")
print(f"Ridge RMSE: {ridge_test_rmse:.3f}")
print(f"ANN RMSE:   {ann_test_rmse:.3f}")
print(f"Baseline:   {baseline_rmse:.3f}")

# --- Optional: bar plot of held-out test RMSE ---
models = ['Ridge', 'ANN', 'Baseline']
rmse_values = [ridge_test_rmse, ann_test_rmse, baseline_rmse]

plt.figure(figsize=(7,5))
bars = plt.bar(models, rmse_values, color=['blue','green','red'])
plt.ylabel('RMSE')
plt.title('Held-out Test Set RMSE Comparison')
plt.grid(axis='y', linestyle='--', alpha=0.7)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.3, f'{height:.2f}',
             ha='center', va='bottom')
plt.show()

NameError: name 'Pipeline' is not defined